# Extração SIH/SUS — Internações Hospitalares · Conecta Saúde

O SIH/SUS é o **numerador** do Índice Composto de Pressão Assistencial. É ele que responde "quantas internações aconteceram" contra o "quantos leitos existem" que vem do CNES.

É também a fonte mais pesada do projeto.

---

## Por que este notebook não guarda os dados brutos

As outras fontes cabem na memória. O SIH não:

| | Um mês de SP | Brasil, 12 meses |
|---|---|---|
| Arquivo `.dbc` | 17 MB | **1.019 MB** |
| Linhas | 225.756 | ~12 milhões |
| Colunas | 113 | 113 |
| Em memória | 250 MB | **~13 GB** |

Concatenar tudo antes de agregar — que é o que a versão anterior fazia — trava a máquina. A solução é inverter a ordem: **agregar arquivo por arquivo e descartar o bruto**, guardando só os resumos. O pico de memória passa a ser o de um arquivo, não o do país inteiro.

O custo real é tempo, não memória: a conversão `.dbc` → DataFrame leva cerca de 90 segundos por arquivo grande, e são 324 arquivos. Reserve **cerca de 3 horas** para o Brasil completo — medido, não estimado. Rodando só São Paulo, são 12 arquivos e cerca de 20 minutos; só o Acre, cerca de 2 minutos.

## 1. Papel do SIH/SUS no projeto

O SIH registra uma linha por **AIH** — Autorização de Internação Hospitalar. Cada AIH é uma internação paga pelo SUS, com procedimento, diagnóstico, tempo de permanência, valor e desfecho.

Cruzamentos previstos:

| Cruzamento | Chave | Indicador resultante |
|---|---|---|
| SIH × CNES | `CNES` | Internações por leito SUS |
| SIH × IBGE | `MUNIC_RES` | Internações por 10 mil habitantes |
| SIH × SIGTAP | `PROC_REA` | Nome e complexidade do procedimento |

### Os dois municípios de cada internação

O RD traz **dois** códigos de município, e confundi-los inverte a conclusão do índice:

| Coluna | Significado |
|---|---|
| `MUNIC_RES` | onde o paciente **mora** |
| `MUNIC_MOV` | onde a internação **aconteceu** |

Quando os dois diferem, houve deslocamento: o paciente não encontrou o serviço na própria cidade. A diferença agregada é o fluxo assistencial — evasão para quem perde pacientes, invasão para quem os recebe.

Isso importa porque os dois denominadores medem coisas diferentes. `MUNIC_RES` cruzado com a população do IBGE mede **necessidade** da população. `MUNIC_MOV` cruzado com os leitos do CNES mede **carga** sobre a estrutura. Um município-polo pequeno pode ter carga altíssima e necessidade baixa — e é exatamente esse o município que o índice precisa destacar.

## 2. Endpoint público

```text
ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/RD{UF}{AA}{MM}.dbc
```

O SIH tem quatro grupos publicados. O projeto usa apenas o `RD`:

| Grupo | Conteúdo | Brasil 2024 |
|---|---|---|
| **`RD`** | **AIH reduzida — uma linha por internação** | **1.019 MB** |
| `SP` | Serviços profissionais, várias linhas por AIH | 3.401 MB |
| `ER` | Erros de processamento | 19 MB |
| `RJ` | AIH rejeitadas | 46 MB |

O `SP` seria necessário para analisar honorários por profissional, o que está fora do escopo do MVP. O `RD` já traz valor total, procedimento e diagnóstico.

In [ ]:
# A ftplib já vem com o Python e não precisa ser instalada.
%pip install -q pandas pyarrow datasus-dbc dbfread

## 3. Parâmetros da extração

In [1]:
from ftplib import FTP
from pathlib import Path
import struct

import pandas as pd

TODAS_AS_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS",
    "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC",
    "SE", "SP", "TO",
]

# ATENÇÃO ao tempo: o Brasil completo leva cerca de 3 horas, porque são
# 324 arquivos e a conversão de cada um é lenta. Para uma prova rápida,
# troque por ["SP"] (cerca de 20 min) ou ["AC"] (cerca de 2 min).
UFS = TODAS_AS_UFS

ANO = 2024
MESES = list(range(1, 13))

FTP_HOST = "ftp.datasus.gov.br"
DIR_SIH = "/dissemin/publicos/SIHSUS/200801_/Dados"
GRUPO = "RD"


def raiz_do_projeto() -> Path:
    '''Devolve a pasta do repositório, subindo até encontrar o .git.'''
    atual = Path.cwd().resolve()
    for pasta in (atual, *atual.parents):
        if (pasta / ".git").exists():
            return pasta
    return atual


RAIZ = raiz_do_projeto()
DIRETORIO_RAW = RAIZ / "dados" / "raw" / "sih"
DIRETORIO_TRATADO = RAIZ / "dados" / "tratado" / "sih"
# Resumos parciais, uma pasta por UF. É o que torna a extração retomável.
DIRETORIO_PARCIAL = DIRETORIO_RAW / "_parciais"

for pasta in (DIRETORIO_RAW, DIRETORIO_TRATADO, DIRETORIO_PARCIAL):
    pasta.mkdir(parents=True, exist_ok=True)


def nome_arquivo(uf: str, ano: int, mes: int) -> str:
    return f"{GRUPO}{uf.upper()}{str(ano)[-2:]}{str(mes).zfill(2)}.dbc"


print(f"Raiz do projeto : {RAIZ}")
print(f"UFs             : {len(UFS)} ({', '.join(UFS[:6])}{'...' if len(UFS) > 6 else ''})")
print(f"Arquivos a ler  : {len(UFS) * len(MESES)}")
print(f"Tempo estimado  : ~{len(UFS) * len(MESES) * 33 / 60:.0f} min")

Raiz do projeto : C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude
UFs             : 27 (AC, AL, AM, AP, BA, CE...)
Arquivos a ler  : 324
Tempo estimado  : ~178 min


## 4. Conexão e conferência de disponibilidade

Mesma abordagem do CNES: uma única listagem do diretório serve todas as UFs, e a conferência roda antes de qualquer download.

In [2]:
def conectar_ftp() -> FTP:
    ftp = FTP(FTP_HOST, timeout=300)
    ftp.login()          # anônimo
    ftp.set_pasv(True)   # obrigatório atrás de firewall/NAT
    return ftp


def conferir_disponibilidade(ufs: list[str], ano: int) -> dict[str, int]:
    '''Conta as competências publicadas de cada UF, numa única listagem.'''
    ftp = conectar_ftp()
    try:
        ftp.cwd(DIR_SIH)
        publicados = set(ftp.nlst())
    finally:
        ftp.quit()

    return {
        uf: sum(1 for a in publicados
                if a.startswith(f"{GRUPO}{uf.upper()}{str(ano)[-2:]}"))
        for uf in ufs
    }


disponiveis = conferir_disponibilidade(UFS, ANO)
print(f"Competências encontradas por UF: {sorted(set(disponiveis.values()))}")

incompletas = {uf: n for uf, n in disponiveis.items() if n < len(MESES)}
if incompletas:
    print(f"\nATENÇÃO — UFs com menos de {len(MESES)} competências: {incompletas}")
else:
    print(f"\nTodas as {len(UFS)} UFs têm as {len(MESES)} competências publicadas.")

Competências encontradas por UF: [12]

Todas as 27 UFs têm as 12 competências publicadas.


## 5. Download, conversão e o recorte de colunas

O `.dbc` é um dBase comprimido com algoritmo proprietário do DATASUS. Fluxo:

```text
.dbc  →  datasus_dbc.decompress  →  .dbf  →  dbfread  →  DataFrame  →  agrega  →  descarta
```

Das 113 colunas do `RD`, o projeto usa 18. O recorte acontece logo após a leitura, antes de qualquer acumulação.

A correção do terminador do DBF vem junto — os arquivos do SIH podem apresentar o mesmo `0x00` no lugar do `0x0D` que quebra a leitura no CNES. A função devolve se precisou corrigir, e o resumo final mostra em quantos arquivos isso aconteceu.

In [3]:
import datasus_dbc
from dbfread import DBF

# 18 das 113 colunas do RD. O que ficou de fora: honorários detalhados,
# diagnósticos secundários 1 a 9, dados de gestor e campos de auditoria.
COLUNAS_RD = [
    "N_AIH",        # identificador da internação
    "MUNIC_RES",    # município de residência do paciente
    "MUNIC_MOV",    # município onde a internação ocorreu
    "CNES",         # estabelecimento
    "PROC_REA",     # procedimento realizado (chave do SIGTAP)
    "DIAG_PRINC",   # diagnóstico principal (CID-10)
    "ESPEC",        # especialidade do leito
    "COMPLEX",      # complexidade
    "CAR_INT",      # caráter da internação: eletiva ou urgência
    "FINANC",       # fonte de financiamento
    "IDADE", "COD_IDADE", "SEXO",
    "DIAS_PERM",    # dias de permanência
    "UTI_MES_TO",   # diárias de UTI
    "MORTE",        # óbito
    "VAL_TOT",      # valor total da AIH
    "DT_INTER",     # data da internação
]

NUMERICAS_RD = ["IDADE", "DIAS_PERM", "UTI_MES_TO", "MORTE", "VAL_TOT"]
TEXTUAIS_RD = ["N_AIH", "MUNIC_RES", "MUNIC_MOV", "CNES", "PROC_REA",
               "DIAG_PRINC", "ESPEC", "COMPLEX", "CAR_INT", "FINANC",
               "COD_IDADE", "SEXO", "DT_INTER"]


def baixar_arquivo(arquivo: str, ftp: FTP | None = None) -> Path:
    '''Baixa um .dbc do FTP, reaproveitando o que já estiver em disco.'''
    destino = DIRETORIO_RAW / arquivo
    if destino.exists() and destino.stat().st_size > 0:
        return destino

    propria = ftp is None
    if propria:
        ftp = conectar_ftp()
        ftp.cwd(DIR_SIH)
    try:
        with open(destino, "wb") as f:
            ftp.retrbinary(f"RETR {arquivo}", f.write, blocksize=65536)
    finally:
        if propria:
            ftp.quit()
    return destino


def corrigir_terminador_dbf(caminho_dbf: Path) -> bool:
    '''Grava o 0x0D que fecha a lista de campos, quando o DATASUS o omite.

    O cabeçalho declara seu próprio tamanho e a lista de campos ocupa 32 bytes
    por campo mais 1 de terminador, então o 0x0D tem de estar em
    `tamanho_do_cabecalho - 1`. Escrever ali não toca em nenhum dado.
    '''
    with open(caminho_dbf, "r+b") as arquivo:
        tam_cabecalho, = struct.unpack("<H", arquivo.read(12)[8:10])
        if (tam_cabecalho - 33) % 32 != 0:
            raise ValueError(f"{caminho_dbf.name}: cabeçalho de {tam_cabecalho} bytes inesperado")
        arquivo.seek(tam_cabecalho - 1)
        if arquivo.read(1) == b"\x0d":
            return False
        arquivo.seek(tam_cabecalho - 1)
        arquivo.write(b"\x0d")
        return True


def ler_dbc(caminho_dbc: Path, limpar: bool = True) -> tuple[pd.DataFrame, bool]:
    '''Converte .dbc em DataFrame já recortado nas colunas de interesse.'''
    caminho_dbf = caminho_dbc.with_suffix(".dbf")
    datasus_dbc.decompress(str(caminho_dbc), str(caminho_dbf))
    corrigido = corrigir_terminador_dbf(caminho_dbf)

    tabela = DBF(str(caminho_dbf), encoding="latin-1", load=False)
    declaradas = len(tabela)
    df = pd.DataFrame(iter(tabela))

    if len(df) != declaradas:
        raise ValueError(f"{caminho_dbc.name}: li {len(df)} de {declaradas} linhas declaradas")

    df = df[[c for c in COLUNAS_RD if c in df.columns]].copy()

    if limpar:
        caminho_dbf.unlink(missing_ok=True)
        caminho_dbc.unlink(missing_ok=True)
    return df, corrigido


def tratar_tipos(df: pd.DataFrame) -> pd.DataFrame:
    '''Códigos como texto, quantidades como número.'''
    for col in TEXTUAIS_RD:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    # Zero à esquerda é significativo: sem o zfill o join com CNES e IBGE falha.
    if "CNES" in df.columns:
        df["CNES"] = df["CNES"].str.zfill(7)
    for col in ("MUNIC_RES", "MUNIC_MOV"):
        if col in df.columns:
            df[col] = df[col].str.zfill(6)
    if "PROC_REA" in df.columns:
        df["PROC_REA"] = df["PROC_REA"].str.zfill(10)

    for col in NUMERICAS_RD:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df

## 6. Agregação em streaming

Esta é a diferença central em relação às outras fontes. Em vez de

```text
ler tudo  →  concatenar  →  agregar        (13 GB de pico)
```

o notebook faz

```text
para cada arquivo:  ler  →  agregar  →  guardar só o resumo  →  descartar
                                                                (250 MB de pico)
```

Cinco grãos de saída, cada um respondendo a uma pergunta diferente:

| Saída | Grão | Para quê |
|---|---|---|
| `residencia` | `MUNIC_RES` × competência | Internações por 10 mil habitantes (com IBGE) |
| `atendimento` | `MUNIC_MOV` × competência | Carga sobre a estrutura local (com CNES) |
| `hospital` | `CNES` × competência | Pressão por estabelecimento |
| `fluxo` | `MUNIC_RES` × `MUNIC_MOV` | Evasão e invasão de pacientes |
| `procedimento` | `PROC_REA` × competência | O que mais interna (com SIGTAP) |

Os resumos parciais são concatenados e reagregados no fim. Isso funciona porque todas as métricas são **aditivas** — somas e contagens se combinam entre partes. Se houvesse mediana ou percentil, este atalho não valeria, porque não se calcula mediana a partir de medianas parciais.

### A extração é retomável

Três horas é tempo suficiente para o notebook ser interrompido: kernel reiniciado, máquina suspensa, queda de rede prolongada. Sem proteção, qualquer uma dessas perde a execução inteira.

Por isso cada UF, ao terminar, grava seus cinco resumos em `dados/raw/sih/_parciais/`. Ao rodar de novo, as UFs já concluídas são lidas do disco em segundos e o download recomeça de onde parou. Os parciais somam poucos megabytes, porque já são agregados.

Para forçar uma extração limpa, apague a pasta `_parciais`.

In [4]:
def agregar_particao(df: pd.DataFrame, competencia: str, uf: str) -> dict[str, pd.DataFrame]:
    '''Reduz um arquivo aos cinco resumos, para o bruto poder ser descartado.'''
    df = df.copy()
    df["COMPETENCIA"] = competencia
    df["UF"] = uf
    df["internacao"] = 1
    df["obito"] = (df["MORTE"] > 0).astype(int)
    df["com_uti"] = (df["UTI_MES_TO"] > 0).astype(int)
    df["urgencia"] = df["CAR_INT"].str.startswith("2").astype(int)

    METRICAS = {
        "internacoes": ("internacao", "sum"),
        "obitos": ("obito", "sum"),
        "internacoes_com_uti": ("com_uti", "sum"),
        "internacoes_urgencia": ("urgencia", "sum"),
        "dias_permanencia": ("DIAS_PERM", "sum"),
        "diarias_uti": ("UTI_MES_TO", "sum"),
        "valor_total": ("VAL_TOT", "sum"),
    }

    return {
        "residencia": df.groupby(["MUNIC_RES", "COMPETENCIA"], as_index=False).agg(**METRICAS),
        "atendimento": df.groupby(["MUNIC_MOV", "UF", "COMPETENCIA"], as_index=False).agg(**METRICAS),
        "hospital": df.groupby(["CNES", "MUNIC_MOV", "UF", "COMPETENCIA"], as_index=False).agg(**METRICAS),
        "fluxo": df.groupby(["MUNIC_RES", "MUNIC_MOV", "COMPETENCIA"], as_index=False).agg(
            internacoes=("internacao", "sum"), valor_total=("VAL_TOT", "sum")
        ),
        "procedimento": df.groupby(["PROC_REA", "COMPETENCIA"], as_index=False).agg(**METRICAS),
    }


CHAVES = {
    "residencia": ["MUNIC_RES", "COMPETENCIA"],
    "atendimento": ["MUNIC_MOV", "UF", "COMPETENCIA"],
    "hospital": ["CNES", "MUNIC_MOV", "UF", "COMPETENCIA"],
    "fluxo": ["MUNIC_RES", "MUNIC_MOV", "COMPETENCIA"],
    "procedimento": ["PROC_REA", "COMPETENCIA"],
}


def caminho_parcial(uf: str, nome: str) -> Path:
    return DIRETORIO_PARCIAL / f"{nome}_{uf}.parquet"


def uf_ja_extraida(uf: str) -> bool:
    '''Uma UF só conta como pronta se os cinco resumos existirem.'''
    return all(caminho_parcial(uf, nome).exists() for nome in CHAVES)


def extrair_sih(ufs: list[str], ano: int, meses: list[int]) -> dict[str, pd.DataFrame]:
    '''Extrai o RD agregando arquivo por arquivo, sem acumular o bruto.

    Retomável: cada UF concluída grava seus resumos em _parciais/, e uma nova
    execução pula o que já está lá.
    '''
    import time

    partes = {nome: [] for nome in CHAVES}
    falhas, total_aih, corrigidos = [], 0, 0
    inicio = time.time()

    pendentes = [uf for uf in ufs if not uf_ja_extraida(uf)]
    prontas = [uf for uf in ufs if uf not in pendentes]
    if prontas:
        print(f"Retomando: {len(prontas)} UF(s) já extraída(s) — {', '.join(prontas)}")
        for uf in prontas:
            for nome in CHAVES:
                partes[nome].append(pd.read_parquet(caminho_parcial(uf, nome)))

    ftp = conectar_ftp()
    ftp.cwd(DIR_SIH)
    try:
        for uf in pendentes:
            desta_uf = {nome: [] for nome in CHAVES}

            for mes in meses:
                arquivo = nome_arquivo(uf, ano, mes)
                try:
                    bruto, corrigido = ler_dbc(baixar_arquivo(arquivo, ftp=ftp))
                    corrigidos += int(corrigido)
                    total_aih += len(bruto)

                    resumos = agregar_particao(
                        tratar_tipos(bruto), f"{ano}{str(mes).zfill(2)}", uf
                    )
                    for nome, parcial in resumos.items():
                        desta_uf[nome].append(parcial)
                    del bruto, resumos          # o bruto não sobrevive ao laço
                except Exception as erro:
                    falhas.append((arquivo, f"{type(erro).__name__}: {erro}"))
                    try:
                        ftp.quit()
                    except Exception:
                        pass
                    ftp = conectar_ftp()
                    ftp.cwd(DIR_SIH)

            # Checkpoint: a UF inteira vira disco antes de seguir para a próxima.
            for nome, lista in desta_uf.items():
                if lista:
                    consolidado = pd.concat(lista, ignore_index=True)
                    consolidado.to_parquet(caminho_parcial(uf, nome), index=False)
                    partes[nome].append(consolidado)

            print(f"  {uf}: {total_aih:,} AIH acumuladas  ({time.time() - inicio:.0f}s)", flush=True)
    finally:
        try:
            ftp.quit()
        except Exception:
            pass

    if falhas:
        print(f"\n{len(falhas)} arquivo(s) falharam:")
        for arquivo, motivo in falhas:
            print(f"  {arquivo}: {motivo}")

    # As métricas são aditivas, então somar os resumos parciais é equivalente
    # a ter agregado tudo de uma vez.
    saida = {}
    for nome, lista in partes.items():
        if not lista:
            saida[nome] = pd.DataFrame()
            continue
        juntos = pd.concat(lista, ignore_index=True)
        soma = {c: "sum" for c in juntos.columns if c not in CHAVES[nome]}
        saida[nome] = juntos.groupby(CHAVES[nome], as_index=False).agg(soma)

    print(f"\nAIH processadas   : {total_aih:,}")
    print(f"Terminador DBF corrigido em {corrigidos} de {len(ufs) * len(meses)} arquivos")
    print(f"Tempo total       : {(time.time() - inicio) / 60:.1f} min")
    return saida


sih = extrair_sih(UFS, ANO, MESES)

for nome, df in sih.items():
    memoria = df.memory_usage(deep=True).sum() / 1024**2 if not df.empty else 0
    print(f"  {nome:<14} {len(df):>9,} linhas   {memoria:>6.1f} MB")

  AC: 57,272 AIH acumuladas  (58s)
  AL: 231,666 AIH acumuladas  (234s)
  AM: 457,669 AIH acumuladas  (435s)
  AP: 514,914 AIH acumuladas  (494s)
  BA: 1,461,578 AIH acumuladas  (1181s)
  CE: 2,048,710 AIH acumuladas  (1594s)
  DF: 2,296,798 AIH acumuladas  (1788s)
  ES: 2,604,298 AIH acumuladas  (2017s)
  GO: 3,056,300 AIH acumuladas  (2343s)
  MA: 3,556,562 AIH acumuladas  (2684s)
  MG: 5,114,641 AIH acumuladas  (3844s)
  MS: 5,329,452 AIH acumuladas  (4012s)
  MT: 5,578,070 AIH acumuladas  (4210s)
  PA: 6,129,124 AIH acumuladas  (4613s)
  PB: 6,390,072 AIH acumuladas  (4818s)
  PE: 7,059,787 AIH acumuladas  (5298s)
  PI: 7,278,961 AIH acumuladas  (5479s)
  PR: 8,316,862 AIH acumuladas  (6260s)
  RJ: 9,219,591 AIH acumuladas  (6918s)
  RN: 9,436,475 AIH acumuladas  (7084s)
  RO: 9,571,219 AIH acumuladas  (7191s)
  RR: 9,608,607 AIH acumuladas  (7227s)
  RS: 10,440,491 AIH acumuladas  (7887s)
  SC: 11,090,659 AIH acumuladas  (8383s)
  SE: 11,211,148 AIH acumuladas  (8478s)
  SP: 13,84

## 7. Validação de qualidade

In [5]:
residencia = sih["residencia"]
atendimento = sih["atendimento"]

total_res = int(residencia["internacoes"].sum())
total_atend = int(atendimento["internacoes"].sum())

validacoes = pd.DataFrame([
    {"verificacao": "Internações (por residência)", "valor": total_res},
    {"verificacao": "Internações (por atendimento)", "valor": total_atend},
    {"verificacao": "Municípios de residência", "valor": int(residencia["MUNIC_RES"].nunique())},
    {"verificacao": "Municípios de atendimento", "valor": int(atendimento["MUNIC_MOV"].nunique())},
    {"verificacao": "Estabelecimentos", "valor": int(sih["hospital"]["CNES"].nunique())},
    {"verificacao": "Procedimentos distintos", "valor": int(sih["procedimento"]["PROC_REA"].nunique())},
    {"verificacao": "Óbitos", "valor": int(residencia["obitos"].sum())},
    {"verificacao": "Competências", "valor": int(residencia["COMPETENCIA"].nunique())},
])
display(validacoes)

# Os dois recortes contam as MESMAS internações, apenas atribuídas a municípios
# diferentes. Se divergirem, alguma agregação perdeu ou duplicou linhas.
assert total_res == total_atend, (
    f"residência ({total_res:,}) e atendimento ({total_atend:,}) deveriam somar igual"
)
print(f"\nOs dois recortes somam as mesmas {total_res:,} internações — agregação consistente.")

print("\nEvolução mensal:")
display(
    residencia.groupby("COMPETENCIA")
    .agg(internacoes=("internacoes", "sum"), obitos=("obitos", "sum"),
         valor_total=("valor_total", "sum"))
    .assign(letalidade_pct=lambda d: (100 * d["obitos"] / d["internacoes"]).round(2))
)

,verificacao,valor
0,Internações (por residência),13945608
1,Internações (por atendimento),13945608
2,Municípios de residência,5570
3,Municípios de atendimento,3134
4,Estabelecimentos,4811
5,Procedimentos distintos,1744
6,Óbitos,610603
7,Competências,12



Os dois recortes somam as mesmas 13,945,608 internações — agregação consistente.

Evolução mensal:


,internacoes,obitos,valor_total,letalidade_pct
COMPETENCIA,,,,
202401,1109094,49851,1.807158e+09,4.49
202402,1094134,47775,1.777413e+09,4.37
202403,1185959,51493,1.918455e+09,4.34
202404,1224862,52748,1.961322e+09,4.31
202405,1219689,53887,1.984013e+09,4.42
202406,1188890,53714,1.970375e+09,4.52
202407,1220080,55584,2.063284e+09,4.56
202408,1229218,54833,2.106841e+09,4.46
202409,1206568,52509,2.065885e+09,4.35


## 8. Fluxo de pacientes — evasão e invasão

A tabela de fluxo permite separar internação **local** de internação **fora do município de residência**. É a medida mais direta de acesso: quando um município evade muito, sua população precisa se deslocar para ser internada.

O cálculo aqui é simples e vale explicitar: uma internação é local quando `MUNIC_RES == MUNIC_MOV`. Tudo o mais é deslocamento.

In [6]:
fluxo = sih["fluxo"].copy()
fluxo["local"] = fluxo["MUNIC_RES"] == fluxo["MUNIC_MOV"]

resumo_fluxo = fluxo.groupby("local")["internacoes"].sum()
total = int(resumo_fluxo.sum())
fora = int(resumo_fluxo.get(False, 0))
print(f"Internações totais            : {total:,}")
print(f"Fora do município de residência: {fora:,} ({100 * fora / total:.1f}%)")

# Evasão por município de residência.
evasao = (
    fluxo.groupby("MUNIC_RES")
    .apply(lambda g: pd.Series({
        "internacoes": g["internacoes"].sum(),
        "fora_do_municipio": g.loc[~g["local"], "internacoes"].sum(),
    }), include_groups=False)
    .reset_index()
)
evasao["taxa_evasao_pct"] = (100 * evasao["fora_do_municipio"] / evasao["internacoes"]).round(1)

print(f"\nMunicípios que evadem 100% das internações: "
      f"{int((evasao['taxa_evasao_pct'] == 100).sum()):,}")
print("\nMunicípios com maior volume evadido:")
display(evasao.nlargest(10, "fora_do_municipio"))

Internações totais            : 13,945,608
Fora do município de residência: 5,047,440 (36.2%)

Municípios que evadem 100% das internações: 2,436

Municípios com maior volume evadido:


,MUNIC_RES,internacoes,fora_do_municipio,taxa_evasao_pct
1552,260790,49482,38547,77.9
1575,260960,30090,25300,84.1
3829,355030,584219,24696,4.2
3113,320130,25381,21630,85.2
3990,410580,21160,18065,85.4
1586,261070,20851,17652,84.7
3224,330350,37617,17630,46.9
3183,330045,26938,17255,64.1
4491,421190,17252,17252,100.0
160,150080,26833,16800,62.6


## 9. Enriquecimento com o SIGTAP

Os procedimentos ganham nome e complexidade. Roda só se o notebook do SIGTAP já tiver sido executado.

O `PROC_REA` foi preenchido com `zfill(10)` na seção 5 justamente para casar com o `CO_PROCEDIMENTO` do SIGTAP, que tem zero à esquerda em todos os 4.844 códigos.

In [7]:
caminho_sigtap = RAIZ / "dados" / "tratado" / "sigtap" / "sigtap_procedimentos_202412.parquet"

if caminho_sigtap.exists():
    sigtap = pd.read_parquet(caminho_sigtap)

    proc = (
        sih["procedimento"]
        .groupby("PROC_REA", as_index=False)
        .agg(internacoes=("internacoes", "sum"), obitos=("obitos", "sum"),
             dias_permanencia=("dias_permanencia", "sum"), valor_total=("valor_total", "sum"))
        .merge(
            sigtap[["CO_PROCEDIMENTO", "NO_PROCEDIMENTO", "NO_GRUPO", "NO_COMPLEXIDADE"]],
            left_on="PROC_REA", right_on="CO_PROCEDIMENTO", how="left",
        )
    )
    sem_nome = proc["NO_PROCEDIMENTO"].isna()
    print(f"Procedimentos sem correspondência no SIGTAP: {int(sem_nome.sum())} de {len(proc)}")
    if sem_nome.any():
        print("  (procedimento desativado entre a competência do SIGTAP e a da produção)")
        print(f"  internações afetadas: {int(proc.loc[sem_nome, 'internacoes'].sum()):,}")

    proc["permanencia_media"] = (proc["dias_permanencia"] / proc["internacoes"]).round(1)

    print("\nOs 15 procedimentos que mais internam:")
    display(
        proc.nlargest(15, "internacoes")[
            ["PROC_REA", "NO_PROCEDIMENTO", "NO_COMPLEXIDADE",
             "internacoes", "permanencia_media", "valor_total"]
        ]
    )

    print("\nInternações por complexidade:")
    display(
        proc.groupby("NO_COMPLEXIDADE")
        .agg(internacoes=("internacoes", "sum"), valor_total=("valor_total", "sum"))
        .sort_values("internacoes", ascending=False)
    )
else:
    print("Execute o notebook do SIGTAP primeiro para ver este enriquecimento.")
    proc = sih["procedimento"]

Procedimentos sem correspondência no SIGTAP: 3 de 1744
  (procedimento desativado entre a competência do SIGTAP e a da produção)
  internações afetadas: 33

Os 15 procedimentos que mais internam:


,PROC_REA,NO_PROCEDIMENTO,NO_COMPLEXIDADE,internacoes,permanencia_media,valor_total
178,0303140151,TRATAMENTO DE PNEUMONIAS OU INFLUENZA (GRIPE),Média complexidade,695778,6.6,9.442135e+08
240,0310010039,PARTO NORMAL,Média complexidade,666007,2.1,3.779320e+08
1471,0411010034,PARTO CESARIANO,Média complexidade,567202,2.6,4.165999e+08
1572,0415010012,TRATAMENTO C/ CIRURGIAS MULTIPLAS,Não se aplica,485624,4.4,1.920338e+09
30,0303010037,TRATAMENTO DE OUTRAS DOENÇAS BACTERIANAS,Média complexidade,425105,10.4,1.475882e+09
25,0301060088,DIAGNOSTICO E/OU ATENDIMENTO DE URGENCIA EM CL...,Média complexidade,292367,1.5,3.605650e+07
144,0303100044,TRATAMENTO DE INTERCORRENCIAS CLINICAS NA GRAV...,Média complexidade,240481,3.6,4.905263e+07
74,0303040149,TRATAMENTO DE ACIDENTE VASCULAR CEREBRAL - AVC...,Média complexidade,228012,7.6,4.125644e+08
183,0303150050,TRATAMENTO DE OUTRAS DOENCAS DO APARELHO URINARIO,Média complexidade,216773,5.6,1.182611e+08
113,0303060212,TRATAMENTO DE INSUFICIENCIA CARDIACA,Média complexidade,203328,8.5,4.394690e+08



Internações por complexidade:


,internacoes,valor_total
NO_COMPLEXIDADE,,
Média complexidade,12173292,1.386899e+10
Alta complexidade,1060693,6.394367e+09
Não se aplica,711590,2.997344e+09


## 10. O indicador central — internações por leito SUS

Aqui o SIH encontra o CNES, e é este número que dá nome ao índice.

O denominador correto é o de **atendimento** (`MUNIC_MOV`), não o de residência: os leitos do CNES ficam onde o hospital está, então a carga que eles suportam é a das internações que ali aconteceram. Usar residência no numerador e leitos no denominador misturaria populações diferentes e produziria razões sem significado nos municípios-polo.

In [8]:
caminho_cnes = RAIZ / "dados" / "tratado" / "cnes" / "cnes_silver_municipio_2024.parquet"

if caminho_cnes.exists():
    leitos = (
        pd.read_parquet(caminho_cnes)
        .groupby("CODUFMUN", as_index=False)
        .agg(leitos_sus=("leitos_sus", "mean"))     # média das 12 competências
    )

    carga = (
        atendimento.groupby("MUNIC_MOV", as_index=False)
        .agg(internacoes=("internacoes", "sum"), diarias_uti=("diarias_uti", "sum"))
        .merge(leitos, left_on="MUNIC_MOV", right_on="CODUFMUN", how="left")
    )

    sem_leito = carga["leitos_sus"].isna() | (carga["leitos_sus"] == 0)
    print(f"Municípios que internaram: {len(carga):,}")
    print(f"  sem leito SUS no CNES  : {int(sem_leito.sum()):,}")
    print("  (internação registrada em município sem leito cadastrado — vale investigar)")

    com_leito = carga[~sem_leito].copy()
    com_leito["internacoes_por_leito"] = (
        com_leito["internacoes"] / com_leito["leitos_sus"]
    ).round(1)

    print(f"\nEntre os {len(com_leito):,} municípios com leito:")
    print(f"  mediana de internações por leito no ano: "
          f"{com_leito['internacoes_por_leito'].median():.1f}")

    grandes = com_leito[com_leito["leitos_sus"] >= 100]
    print(f"\nMaior pressão entre os {len(grandes):,} municípios com 100+ leitos:")
    display(grandes.nlargest(15, "internacoes_por_leito")[
        ["MUNIC_MOV", "internacoes", "leitos_sus", "internacoes_por_leito"]
    ])
else:
    print("Execute o notebook do CNES primeiro para ver este indicador.")

Municípios que internaram: 3,134
  sem leito SUS no CNES  : 0
  (internação registrada em município sem leito cadastrado — vale investigar)

Entre os 3,134 municípios com leito:
  mediana de internações por leito no ano: 27.1

Maior pressão entre os 527 municípios com 100+ leitos:


,MUNIC_MOV,internacoes,leitos_sus,internacoes_por_leito
715,231330,14097,143.000000,98.6
1755,316800,10048,108.833333,92.3
2413,412550,13247,145.500000,91.0
887,260120,14892,164.333333,90.6
2264,410400,36035,400.000000,90.1
2302,410840,24036,270.166667,89.0
2452,420200,10322,118.500000,87.1
2114,353470,11370,131.333333,86.6
2133,353870,36372,426.916667,85.2
2369,411850,17338,205.666667,84.3


## 11. Salvamento em Parquet

Saída em `dados/tratado/sih/`. Todas as tabelas já são agregadas, então são pequenas — o bruto de 13 GB nunca chega ao disco.

In [9]:
arquivos_gerados = []


def salvar(df: pd.DataFrame, nome: str) -> None:
    if df.empty:
        return
    destino = DIRETORIO_TRATADO / nome
    df.to_parquet(destino, index=False)
    arquivos_gerados.append(destino)


for nome, df in sih.items():
    salvar(df, f"sih_silver_{nome}_{ANO}.parquet")

if caminho_sigtap.exists():
    salvar(proc, f"sih_procedimentos_enriquecidos_{ANO}.parquet")

total_mb = sum(a.stat().st_size for a in arquivos_gerados) / 1024**2
print(f"{len(arquivos_gerados)} arquivos, {total_mb:.1f} MB")
print(f"Pasta: {DIRETORIO_TRATADO}\n")
for arq in sorted(arquivos_gerados):
    print(f"  {arq.name:<46} {arq.stat().st_size / 1024:>9,.1f} KB")

6 arquivos, 6.7 MB
Pasta: C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude\dados\tratado\sih

  sih_procedimentos_enriquecidos_2024.parquet        102.8 KB
  sih_silver_atendimento_2024.parquet                524.1 KB
  sih_silver_fluxo_2024.parquet                    4,023.5 KB
  sih_silver_hospital_2024.parquet                   808.2 KB
  sih_silver_procedimento_2024.parquet               324.5 KB
  sih_silver_residencia_2024.parquet               1,038.0 KB


## 12. Perguntas que o SIH/SUS responde

1. Quantas internações SUS cada município gera e quantas recebe?
2. Quais municípios dependem inteiramente da rede de outros para internar?
3. Quais hospitais concentram a maior carga assistencial?
4. Quais procedimentos mais pressionam a rede, e qual sua complexidade?
5. Qual a permanência média e a letalidade por procedimento?
6. Quantas internações cada leito SUS absorve no ano?

Cruzado com CNES e IBGE:

```text
Internações por leito SUS    = internações SIH (MUNIC_MOV) / leitos SUS CNES
Internações por 10 mil hab.  = (internações SIH (MUNIC_RES) / população IBGE) * 10.000
Taxa de evasão               = internações fora do município / internações do município
```

As três medem coisas distintas, e a escolha do município — residência ou atendimento — é o que separa necessidade de carga.